In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import mne
import torch
from pathlib import Path
from dn3_ext import BENDRClassification

data_dir = Path("data")
edf_path_demo = data_dir / "our_demo" / "test_from_20s.edf"

def load_modules(mdl: BENDRClassification, enc_weights: Path, cont_weights: Path) -> None:
    device = "cpu" # TODO if torch.cuda.is_available, etc

    enc_sd = torch.load(str(enc_weights), map_location=device)
    mdl.encoder.load_state_dict(enc_sd, strict=False)

    ctx_sd = torch.load(str(cont_weights), map_location=device)
    mdl.contextualizer.load_state_dict(ctx_sd, strict=False)


In [ ]:
raw_demo = mne.io.read_raw_edf(edf_path_demo, preload=True, verbose=False)
raw_demo.drop_channels(['ECG  ECG', 'EEG OZ-A1']) # TODO convert to meaningfull channels
weights_dir = Path("pretrained_weights")

In [ ]:
arr: np.ndarray = raw_demo.get_data()  # shape: channels x samples
x = torch.from_numpy(arr[np.newaxis, ...]).float()  # shape: batch x channels x samples

tueg_model = BENDRClassification(targets=2, samples=arr.shape[1], channels=arr.shape[0])
# model.load_pretrained_modules(weights_dir / "encoder.pt",
#                               weights_dir / "contextualizer.pt",
#                               strict=False)
load_modules(tueg_model, weights_dir / "encoder.pt", weights_dir / "contextualizer.pt")

# make sure `mask_replacement` is not trainable
if hasattr(tueg_model.contextualizer, "mask_replacement"):
    tueg_model.contextualizer.mask_replacement.requires_grad = False

tueg_model.eval()
with torch.no_grad():
    feats = tueg_model.features_forward(x)

print("feature shape:", feats.shape)
print("first values:", feats.flatten()[:10])

# Load an example from TUEV

In [ ]:
mne.io.read_raw_edf(data_dir / "tueg_demo" / "aaaaaeri_00000001.edf").ch_names

In [ ]:
raw_demo.ch_names

In [ ]:
from dn3.dn3.configuratron import ExperimentConfig
from dn3.dn3.transforms.instance import To1020

tueg_config_path = Path("configs") / "tueg_local_demo.yml"
tueg_experiment_config = ExperimentConfig(str(tueg_config_path))
tueg_config = tueg_experiment_config.datasets['tueg']
tueg_dataset = tueg_config.auto_construct_dataset()
tueg_dataset.add_transform(To1020())

In [ ]:
from dn3_ext import BENDRClassification

tueg_model = BENDRClassification.from_dataset(tueg_dataset)
load_modules(tueg_model, Path(tueg_experiment_config.encoder_weights), Path(tueg_experiment_config.context_weights)) # TODO freeze encoder

In [ ]:
import torch
from dn3.dn3.trainable.processes import StandardClassification

process = StandardClassification(tueg_model, metrics=['Accuracy'])
process.set_optimizer(torch.optim.AdamW(process.parameters(), tueg_config.lr, weight_decay=0.01))

In [ ]:
# TODO add classes to the dataset
# FIXME fails here: dataset iterator yields `inputs` of length 1, model takes inputs[:-1]
process.fit(training_dataset=tueg_dataset)

# Load our data using D3N tools

In [ ]:
from dn3.dn3.configuratron import ExperimentConfig
from dn3.dn3.transforms.instance import To1020
from dn3_ext import BENDRClassification

ours_config_path = Path("configs") / "ours_local_demo.yml"
ours_experiment_config = ExperimentConfig(str(ours_config_path))
ours_data_config = ours_experiment_config.datasets['demo']
ours_demo_dataset = ours_data_config.auto_construct_dataset()
ours_demo_dataset.add_transform(To1020())

model_ours = BENDRClassification.from_dataset(ours_demo_dataset)
load_modules(model_ours, Path(ours_experiment_config.encoder_weights), Path(ours_experiment_config.context_weights))